# TRIBE v2 full mode (audio + text) on a Colab GPU

Runs the text-pathway experiment from [fusi-tribe-speech](https://github.com/saanviiyer/fusi-tribe-speech)
on a free Colab GPU. On a laptop without an NVIDIA GPU, Llama 3.2 word embeddings are too slow and memory-hungry.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Have a Hugging Face token whose account has been granted access to
   [meta-llama/Llama-3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B).
3. Have `stimuli_for_colab.zip`: the six stimulus WAVs and `manifest.json` from `outputs/stimuli/`
   (`cd outputs && zip -r ../stimuli_for_colab.zip stimuli -x 'stimuli/passages/*' -x 'stimuli/*.tsv'`).

Takes about 1 to 2 hours on a T4. At the end, download `full_mode_results.zip` and unzip it into the repo's
`outputs/` folder, then run `fusi-tribe analyze-tribe --mode full` and `fusi-tribe report` locally.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install
After this cell finishes, use *Runtime → Restart session* once (TRIBE pins numpy 2.2.6), then continue with step 2.

In [ ]:
%cd /content
!git clone -q https://github.com/saanviiyer/fusi-tribe-speech
%cd /content/fusi-tribe-speech
!pip install -q -e ".[tribe]"

## 2. Log in to Hugging Face
Paste your token when asked. It stays in this Colab session.

In [ ]:
%cd /content/fusi-tribe-speech
from huggingface_hub import login, auth_check
login()
auth_check("meta-llama/Llama-3.2-3B")   # raises if the account has no access yet
print("Llama 3.2 access OK")

## 3. Upload the stimuli
Choose `stimuli_for_colab.zip`.

In [ ]:
import zipfile, json, hashlib, pathlib
from google.colab import files
uploaded = files.upload()
zipfile.ZipFile(next(iter(uploaded))).extractall("outputs")
manifest = json.loads(pathlib.Path("outputs/stimuli/manifest.json").read_text())
for cond, info in manifest["conditions"].items():
    md5 = hashlib.md5(pathlib.Path("outputs/stimuli", info["path"]).read_bytes()).hexdigest()
    assert md5 == info["md5"], f"{cond}: checksum mismatch"
print("All", len(manifest["conditions"]), "stimuli match their checksums")

## 4. Run TRIBE in full mode
Transcribes each run with WhisperX, embeds the words with Llama 3.2, and predicts the cortical response.

In [ ]:
!fusi-tribe transcribe
!fusi-tribe run-tribe --mode full

## 5. Download the results

In [ ]:
!cd outputs && zip -qr ../full_mode_results.zip predictions/full transcripts
from google.colab import files
files.download("full_mode_results.zip")